# Rental Price Prediction — XGB + LGB + CatBoost Ensemble
**Runtime:** GPU — Runtime → Change runtime type → T4 GPU

Загрузите `train.csv` и `test.csv` через панель файлов слева (иконка папки), затем запускайте ячейки по порядку.

In [ ]:
!pip install -q optuna lightgbm xgboost catboost
!pip install optuna-integration[catboost]


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import RidgeCV
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import optuna
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

HAS_GPU = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
print(f'GPU available: {HAS_GPU}')

N_TRIALS = 150
N_SPLITS = 5
SEED = 42

FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

TARGET = 'rental_price'
ID_COL = 'id'

X      = train.drop(columns=[ID_COL, TARGET])
y      = train[TARGET]
X_test = test.drop(columns=[ID_COL])

print(f'Train: {X.shape}, Test: {X_test.shape}')

In [ ]:
missing_cols = ['battery_level_start', 'temperature_c', 'wind_speed', 'route_complexity']

def add_features(df):
    df = df.copy()

    # --- Missing flags ---
    for col in missing_cols:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    # --- Category interaction ---
    df['zone_x_model'] = df['city_zone'] + '_' + df['scooter_model']

    # --- Base ratios ---
    df['speed_kmh']              = df['distance_km'] / (df['trip_duration_min'] / 60 + 1e-6)
    df['price_per_km_est']       = df['avg_price_last_week'] / (df['distance_km'] + 1e-6)
    df['price_per_min_est']      = df['avg_price_last_week'] / (df['trip_duration_min'] + 1e-6)

    # --- Demand interactions ---
    df['demand_weekend']         = df['demand_index'] * df['is_weekend']
    df['duration_x_demand']      = df['trip_duration_min'] * df['demand_index']
    df['distance_x_demand']      = df['distance_km'] * df['demand_index']
    df['avg_price_x_demand']     = df['avg_price_last_week'] * df['demand_index']

    # --- Noisy distance ---
    df['dist_noisy_diff']        = df['distance_km_noisy'] - df['distance_km']
    df['dist_noisy_ratio']       = df['distance_km_noisy'] / (df['distance_km'] + 1e-6)

    # --- Battery ---
    df['battery_x_distance']     = df['battery_level_start'] * df['distance_km']
    df['battery_per_km']         = df['battery_level_start'] / (df['distance_km'] + 1e-6)

    # --- Weather ---
    df['temp_x_wind']            = df['temperature_c'] * df['wind_speed']
    df['weather_x_demand']       = df['weather_rating'] * df['demand_index']

    # --- Route & experience ---
    df['duration_x_route']       = df['trip_duration_min'] * df['route_complexity']
    df['distance_x_route']       = df['distance_km'] * df['route_complexity']
    df['experience_x_demand']    = df['driver_experience'] * df['demand_index']
    df['experience_x_distance']  = df['driver_experience'] * df['distance_km']
    df['experience_x_route']     = df['driver_experience'] * df['route_complexity']

    # --- Squares ---
    df['demand_sq']              = df['demand_index'] ** 2
    df['distance_sq']            = df['distance_km'] ** 2
    df['duration_sq']            = df['trip_duration_min'] ** 2
    df['avg_price_sq']           = df['avg_price_last_week'] ** 2

    # --- Logs ---
    df['log_distance']           = np.log1p(df['distance_km'])
    df['log_duration']           = np.log1p(df['trip_duration_min'])
    df['log_avg_price']          = np.log1p(df['avg_price_last_week'])

    return df


X      = add_features(X)
X_test = add_features(X_test)

# --- Target encoding (CV-safe: fold-based for train, global mean for test) ---
te_cols = ['city_zone', 'scooter_model', 'zone_x_model']

for col in te_cols:
    global_mean = y.mean()
    te_train = pd.Series(np.nan, index=X.index)
    for train_idx, val_idx in kf.split(X):
        means = y.iloc[train_idx].groupby(X[col].iloc[train_idx]).mean()
        te_train.iloc[val_idx] = X[col].iloc[val_idx].map(means)
    te_train = te_train.fillna(global_mean)
    X[f'{col}_te'] = te_train.values

    te_test = X_test[col].map(y.groupby(X[col]).mean()).fillna(global_mean)
    X_test[f'{col}_te'] = te_test.values

cat_cols = ['city_zone', 'scooter_model', 'zone_x_model']
num_cols = [c for c in X.columns if c not in cat_cols]

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
])

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

X_proc      = preprocessor.fit_transform(X)
X_test_proc = preprocessor.transform(X_test)

all_cols = num_cols + cat_cols
cat_feature_indices = [all_cols.index(c) for c in cat_cols]

for idx in cat_feature_indices:
    X_proc[:, idx]      = X_proc[:, idx].astype(int)
    X_test_proc[:, idx] = X_test_proc[:, idx].astype(int)

print(f'Features after engineering: {X_proc.shape[1]}')

In [ ]:
cb_task_type = 'GPU' if HAS_GPU else 'CPU'
CB_TRIALS = 50

# CatBoost requires object dtype array when cat_features are specified
X_proc_cb      = X_proc.astype(object)
X_test_proc_cb = X_test_proc.astype(object)
for idx in cat_feature_indices:
    X_proc_cb[:, idx]      = X_proc[:, idx].astype(int)
    X_test_proc_cb[:, idx] = X_test_proc[:, idx].astype(int)

print(f'CatBoost: {CB_TRIALS} trials, task_type={cb_task_type}')


def tune_cb(X_cb, y_vals, kf, n_trials, cat_indices):
    def cb_cv_inner(params):
        oof = np.zeros(len(y_vals))
        for train_idx, val_idx in kf.split(X_cb):
            model = CatBoostRegressor(**params)
            model.fit(X_cb[train_idx], y_vals.iloc[train_idx], cat_features=cat_indices)
            oof[val_idx] = model.predict(X_cb[val_idx])
        return 1 - np.sum((y_vals - oof)**2) / np.sum((y_vals - y_vals.mean())**2)

    def objective(trial):
        params = dict(
            iterations          = trial.suggest_int('iterations', 300, 1000),
            learning_rate       = trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
            depth               = trial.suggest_int('depth', 4, 10),
            l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1e-3, 50.0, log=True),
            bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 5.0),
            random_strength     = trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
            border_count        = trial.suggest_int('border_count', 32, 255),
            task_type=cb_task_type, random_state=SEED, verbose=0,
        )
        r2 = cb_cv_inner(params)
        print(f'  Trial {trial.number}/{n_trials}: R²={r2:.6f}', flush=True)
        return r2

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    print(f'\nCatBoost best R² = {study.best_value:.6f}')
    return study.best_params, study.best_value


cb_best_params, cb_best_r2 = tune_cb(X_proc_cb, y, kf, CB_TRIALS, cat_feature_indices)

In [ ]:
from catboost import CatBoostRegressor, Pool

print('\nGenerating CatBoost OOF predictions...')

oof_cb = np.zeros(len(y))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_cb)):

    X_train = X_cb.iloc[train_idx]
    X_val   = X_cb.iloc[val_idx]
    y_train = y.iloc[train_idx]
    y_val   = y.iloc[val_idx]

    train_pool = Pool(X_train, y_train, cat_features=cat_cols)
    val_pool   = Pool(X_val, y_val, cat_features=cat_cols)

    model = CatBoostRegressor(
        **cb_best_params,
        task_type=cb_task_type,
        random_state=SEED,
        verbose=0
    )

    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=50,
        use_best_model=True,
        verbose=0
    )

    oof_cb[val_idx] = model.predict(X_val)

    print(f"Fold {fold+1}/{kf.get_n_splits()} done")

In [ ]:
print('Training final CatBoost model on full data...')

best_cb = CatBoostRegressor(
    **cb_best_params,
    task_type=cb_task_type,
    random_state=SEED,
    verbose=0
)

best_cb.fit(
    X_cb,
    y,
    cat_features=cat_cols,
    verbose=0
)

final_pred = best_cb.predict(X_test_cb)


submission = pd.DataFrame({
    'id': test[ID_COL],
    'rental_price': final_pred
})

submission.to_csv('submission.csv', index=False)

print(f'Submission saved: {len(submission)} rows')
submission.head()

In [ ]:
from google.colab import files
files.download('submission.csv')